In [68]:
import numpy as np
import pandas as pd

In [69]:
filename = "../Data/Health_Risk_Dataset.csv"
df = pd.read_csv(filename)
df = df.drop(['Patient_ID'], axis = 1)
df = df[df['Risk_Level'] != 'Low']
df.head()

,Respiratory_Rate,Oxygen_Saturation,O2_Scale,Systolic_BP,Heart_Rate,Temperature,Consciousness,On_Oxygen,Risk_Level
0,25,96,1,97,107,37.5,A,0,Medium
1,28,92,2,116,151,38.5,P,1,High
2,29,91,1,79,135,38.4,A,0,High
3,24,96,1,95,92,37.3,A,0,Medium
5,20,91,2,121,98,37.8,P,1,Medium


# EDA and Preprocessing

In [70]:
print(df.shape)

(745, 9)


In [71]:
df.describe()

,Respiratory_Rate,Oxygen_Saturation,O2_Scale,Systolic_BP,Heart_Rate,Temperature,On_Oxygen
count,745.000000,745.000000,745.000000,745.000000,745.000000,745.000000,745.000000
mean,22.535570,91.779866,1.166443,102.970470,101.632215,38.173154,0.330201
std,5.600176,4.864273,0.372728,18.911983,21.357971,1.141673,0.470601
min,12.000000,74.000000,1.000000,50.000000,60.000000,35.800000,0.000000
25%,18.000000,89.000000,1.000000,89.000000,85.000000,37.200000,0.000000
50%,22.000000,92.000000,1.000000,105.000000,100.000000,38.100000,0.000000
75%,27.000000,96.000000,1.000000,117.000000,116.000000,38.900000,1.000000
max,40.000000,100.000000,2.000000,146.000000,163.000000,41.800000,1.000000


In [72]:
df.info()

<class 'pandas.DataFrame'>
Index: 745 entries, 0 to 999
Data columns (total 9 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   Respiratory_Rate   745 non-null    int64  
 1   Oxygen_Saturation  745 non-null    int64  
 2   O2_Scale           745 non-null    int64  
 3   Systolic_BP        745 non-null    int64  
 4   Heart_Rate         745 non-null    int64  
 5   Temperature        745 non-null    float64
 6   Consciousness      745 non-null    str    
 7   On_Oxygen          745 non-null    int64  
 8   Risk_Level         745 non-null    str    
dtypes: float64(1), int64(6), str(2)
memory usage: 62.9 KB


## Encode Risk_Level

In [73]:
# from sklearn.preprocessing import LabelEncoder

# encoder = LabelEncoder() 

# df["Risk_Level"] = encoder.fit_transform(df["Risk_Level"])

# mapping = dict(zip(encoder.classes_, range(len(encoder.classes_))))
# print(mapping)

In [74]:
risk_mapping = {
    'Normal': np.int64(0),
    'Medium': np.int64(1),
    'High': np.int64(2)
}

# 2. Map the dictionary to your column
df['Risk_Level'] = df['Risk_Level'].map(risk_mapping)

## Drop features not on sensor

In [75]:
df_for_sensor = df[['Oxygen_Saturation', 'Heart_Rate', 'Temperature', 'Risk_Level']]
df_for_sensor.head()

,Oxygen_Saturation,Heart_Rate,Temperature,Risk_Level
0,96,107,37.5,1
1,92,151,38.5,2
2,91,135,38.4,2
3,96,92,37.3,1
5,91,98,37.8,1


In [76]:
df.head()

,Respiratory_Rate,Oxygen_Saturation,O2_Scale,Systolic_BP,Heart_Rate,Temperature,Consciousness,On_Oxygen,Risk_Level
0,25,96,1,97,107,37.5,A,0,1
1,28,92,2,116,151,38.5,P,1,2
2,29,91,1,79,135,38.4,A,0,2
3,24,96,1,95,92,37.3,A,0,1
5,20,91,2,121,98,37.8,P,1,1


## Split X and y

In [77]:
X = df_for_sensor.drop(labels = ['Risk_Level'], axis = 1)
y = df_for_sensor['Risk_Level']

# Modelling

In [78]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.2, random_state = 42, stratify = y)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [79]:
y_train.value_counts()

Risk_Level
1    245
2    223
0    128
Name: count, dtype: int64

In [80]:
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, roc_curve, auc

from sklearn.model_selection import StratifiedKFold, cross_val_score
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.neural_network import MLPClassifier

In [81]:
# cross validation to see if any model overfits
def cross_validate(model, X, y):
    kfold = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    pipeline = Pipeline(steps=[('s', StandardScaler()), ('o', SMOTE()), ('m', model)])
    scores = cross_val_score(pipeline, X, y, scoring='accuracy', cv=kfold, n_jobs=-1)
    return scores

In [82]:

models = []
models.append(('Support Vector Machine', SVC()))
models.append(('Decision Tree', DecisionTreeClassifier()))
models.append(('ANN', MLPClassifier(max_iter=1000, random_state=42)))

results = []
names = []

for name, model in models:
    scores = cross_validate(model, X_train, y_train)
    results.append(scores)
    names.append(name)
    print(f'{name}: {scores.mean()} +- ({scores.std()})')
    model.fit(X_train, y_train)

Support Vector Machine: 0.947983193277311 +- (0.016304253568162612)
Decision Tree: 0.9144397759103642 +- (0.018662295499043833)
ANN: 0.9395938375350139 +- (0.017150293145680572)


In [85]:
model = SVC()

model.fit(X_train, y_train)
y_pred = model.predict(X_test)

cm = confusion_matrix(y_test, y_pred)
print(cm)

print("\nClassification Report:\n", classification_report(y_test, y_pred))

# Individual Scores
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred, average='weighted')  # Change to 'macro' or 'micro' if needed
recall = recall_score(y_test, y_pred, average='weighted')
f1 = f1_score(y_test, y_pred, average='weighted')

print(f"Accuracy: {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall: {recall:.4f}")
print(f"F1 Score: {f1:.4f}")

[[32  0  0]
 [ 2 58  1]
 [ 0  4 52]]

Classification Report:
               precision    recall  f1-score   support

           0       0.94      1.00      0.97        32
           1       0.94      0.95      0.94        61
           2       0.98      0.93      0.95        56

    accuracy                           0.95       149
   macro avg       0.95      0.96      0.96       149
weighted avg       0.95      0.95      0.95       149

Accuracy: 0.9530
Precision: 0.9539
Recall: 0.9530
F1 Score: 0.9530


In [86]:
import pickle

# Save the model to a file
with open('sensor_model.pkl', 'wb') as f:
    pickle.dump(model, f)

# # Load the model back later
# with open('my_model.pkl', 'rb') as f:
#     loaded_model = pickle.load(f)